# 04 · Temporal Analysis
## How Illicit Activity Evolves Across 49 Time Steps

### Prerequisites
Run notebooks 01, 02, and 03 first. The following must be in memory or on disk:
- `df` — merged DataFrame (from notebook 01)
- `G` — NetworkX graph (from notebook 01)
- `full_features.csv` — produced by notebook 02
- `unknown_node_predictions.csv` — produced by notebook 03

---

### What this notebook does
The Elliptic dataset's 49 time steps enable studying the **dynamics of financial crime** — not just who is fraudulent, but when, how fast, and in what pattern illicit activity emerges.

Four original research questions are answered:
1. **Burst detection** — At which time steps is illicit activity statistically anomalous?
2. **Propagation** — Does illicit activity spread to neighbouring nodes over subsequent time steps?
3. **Early warning** — Can we predict an upcoming burst before it peaks?
4. **Combined timeline** — What does the full picture look like with GNN predictions overlaid?

### Key findings
- 6 burst windows: steps 9, 13, 20, 28, 29, 32
- Mean propagation rate: 35.57% (3.5× the base rate)
- Peak propagation: 84.11% at step 29
- 1,971% increase in detectable suspicious activity when GNN predictions are added

In [ ]:
# Imports and load all required data
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'#0D1117', 'axes.facecolor':'#161B22',
    'axes.edgecolor':'#30363D',   'axes.labelcolor':'#C9D1D9',
    'xtick.color':'#8B949E',      'ytick.color':'#8B949E',
    'text.color':'#C9D1D9',       'grid.color':'#21262D',
    'grid.linestyle':'--',        'figure.dpi':130,
})

full_df    = pd.read_csv('full_features.csv')
unknown_df = pd.read_csv('unknown_node_predictions.csv')

print(f'Full feature matrix : {full_df.shape}')
print(f'Unknown predictions : {unknown_df.shape}')
print(f'\nStarting Temporal Analysis...')
print(f'Confirmed illicit : {(full_df["class"]==1).sum():,}')
print(f'GNN predicted illicit : {(unknown_df["predicted_label"]=="Illicit").sum():,}')

## 1. Burst Detection
Uses z-score anomaly detection to identify time steps where illicit activity is
statistically anomalous — not just visually high, but significantly above the mean.

In [ ]:
# Compute illicit rate at each time step and flag burst windows using z-score
ts_stats = full_df.groupby('time_step').apply(lambda g: pd.Series({
    'total'        : len(g),
    'illicit'      : (g['class'] == 1).sum(),
    'licit'        : (g['class'] == 2).sum(),
    'unknown'      : (g['class'] == 0).sum(),
    'illicit_rate' : (g['class'] == 1).sum() / max((g['class'].isin([1,2])).sum(), 1),
})).reset_index()

mean_r = ts_stats['illicit_rate'].mean()
std_r  = ts_stats['illicit_rate'].std()
ts_stats['z_score']  = (ts_stats['illicit_rate'] - mean_r) / std_r
ts_stats['is_burst'] = ts_stats['z_score'] > 1.5

burst_steps = ts_stats[ts_stats['is_burst']]['time_step'].tolist()
print(f'Burst time steps (z > 1.5): {burst_steps}')
print(f'Mean illicit rate : {mean_r:.4f}')
print(f'Std illicit rate  : {std_r:.4f}')

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle('Temporal Burst Detection — Illicit Activity Across 49 Time Steps', fontsize=14)

axes[0].bar(ts_stats['time_step'], ts_stats['licit'],
            color='#00C9A7', alpha=0.7, label='Licit')
axes[0].bar(ts_stats['time_step'], ts_stats['unknown'],
            color='#8B949E', alpha=0.5, label='Unknown',
            bottom=ts_stats['licit'])
axes[0].bar(ts_stats['time_step'], ts_stats['illicit'],
            color='#FF4444', alpha=0.9, label='Illicit',
            bottom=ts_stats['licit'] + ts_stats['unknown'])
axes[0].set_ylabel('Node Count')
axes[0].legend(loc='upper right')
axes[0].set_title('All Classes per Time Step')

burst_colours = ts_stats['is_burst'].map({True: '#FF4444', False: '#4A90D9'})
axes[1].bar(ts_stats['time_step'], ts_stats['illicit_rate'],
            color=burst_colours, alpha=0.85, width=0.8)
axes[1].axhline(mean_r, color='#FFD700', ls='--', lw=1.5,
                label=f'Mean: {mean_r:.3f}')
axes[1].axhline(mean_r + 1.5*std_r, color='#FF4444', ls=':', lw=1,
                label='Burst threshold (1.5σ)')
axes[1].set_ylabel('Illicit Rate (among labelled)')
axes[1].set_xlabel('Time Step')
axes[1].set_title('Illicit Rate — Red Bars = Statistically Anomalous Bursts')
axes[1].legend()

for _, row in ts_stats[ts_stats['is_burst']].iterrows():
    axes[1].annotate(f"t={int(row['time_step'])}",
                     xy=(row['time_step'], row['illicit_rate']),
                     xytext=(0, 8), textcoords='offset points',
                     ha='center', fontsize=7, color='#FF4444')

plt.tight_layout()
plt.savefig('fig_burst_detection.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Burst detection chart saved')

## 2. Propagation Analysis
For each time step T, measures what proportion of nodes that received funds from
illicit sources at T were themselves illicit at T+1.

In [ ]:
# Compute propagation rate across consecutive time steps
print('Computing propagation rates...')

label_map_class = full_df.set_index('txId')['class'].to_dict()
label_map_ts    = full_df.set_index('txId')['time_step'].to_dict()
records         = []

for t in range(1, 49):
    ill_t = set(full_df[(full_df['time_step']==t) & (full_df['class']==1)]['txId'])
    if len(ill_t) == 0: continue

    exposed_t1 = set()
    for n in ill_t:
        if n in G:
            for succ in G.successors(n):
                if succ in label_map_ts:
                    exposed_t1.add(succ)

    if len(exposed_t1) == 0: continue

    n_ill_next = sum(1 for n in exposed_t1 if label_map_class.get(n, 0) == 1)
    records.append({
        'time_step'        : t,
        'illicit_at_t'     : len(ill_t),
        'exposed_at_t1'    : len(exposed_t1),
        'illicit_at_t1'    : n_ill_next,
        'propagation_rate' : n_ill_next / len(exposed_t1),
    })

prop_df = pd.DataFrame(records)
print(f'Mean propagation rate : {prop_df["propagation_rate"].mean()*100:.2f}%')
peak    = prop_df.loc[prop_df['propagation_rate'].idxmax()]
print(f'Peak propagation      : {peak["propagation_rate"]*100:.2f}% at t={int(peak["time_step"])}')

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(prop_df['time_step'], prop_df['propagation_rate'],
       color='#FF4444', alpha=0.8, width=0.7)
ax.axhline(prop_df['propagation_rate'].mean(), color='#FFD700', ls='--', lw=1.5,
           label=f'Mean: {prop_df["propagation_rate"].mean()*100:.2f}%')
ax.set_title('Illicit Propagation Rate — Proportion of Exposed Nodes\n'
             'that Become Illicit at the Next Time Step', fontsize=12)
ax.set_xlabel('Time Step T')
ax.set_ylabel('Propagation Rate')
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=1))
ax.legend()
plt.tight_layout()
plt.savefig('fig_propagation.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Propagation chart saved')

## 3. Early Warning Model
Can we predict the NEXT time step's illicit rate from recent history?
Uses a simple autoregressive Ridge regression with 3 lagged features.

In [ ]:
# Build and evaluate an early warning model for illicit rate prediction
ts_model = ts_stats[['time_step', 'illicit_rate', 'total', 'illicit']].copy()
for lag in [1, 2, 3]:
    ts_model[f'ill_rate_lag{lag}'] = ts_model['illicit_rate'].shift(lag)
    ts_model[f'total_lag{lag}']    = ts_model['total'].shift(lag)
ts_model = ts_model.dropna().reset_index(drop=True)

lag_cols = [c for c in ts_model.columns if 'lag' in c]
X_ts     = ts_model[lag_cols].values
y_ts     = ts_model['illicit_rate'].values

split = len(X_ts) - 10
X_tr, X_te = X_ts[:split], X_ts[split:]
y_tr, y_te = y_ts[:split], y_ts[split:]

ew_model  = Ridge(alpha=1.0)
ew_model.fit(X_tr, y_tr)
y_pred_ts = ew_model.predict(X_te)
mae       = mean_absolute_error(y_te, y_pred_ts)

print(f'Early warning MAE   : {mae:.5f}')
print(f'Mean illicit rate   : {y_ts.mean():.5f}')
print(f'MAE as % of mean    : {mae/y_ts.mean()*100:.1f}%')

fig, ax = plt.subplots(figsize=(13, 4))
all_ts = ts_model['time_step'].values
ax.plot(all_ts, y_ts, color='#4A90D9', lw=1.5, label='Actual illicit rate')
ax.plot(all_ts[split:], y_pred_ts, color='#FFD700', lw=2, ls='--',
        label=f'Predicted (MAE={mae:.4f})')
ax.axvline(all_ts[split], color='#8B949E', ls=':', lw=1.5,
           label='Train / test boundary')
ax.set_title('Early Warning Model — Predicting Next Time Step Illicit Rate', fontsize=12)
ax.set_xlabel('Time Step')
ax.set_ylabel('Illicit Rate')
ax.legend()
plt.tight_layout()
plt.savefig('fig_early_warning.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Early warning chart saved')

## 4. Combined Confirmed + GNN-Predicted Timeline
The key research contribution visualised — showing how many more at-risk nodes
we identify by adding GNN predictions on top of the confirmed labels.

In [ ]:
# Overlay confirmed and GNN-predicted illicit nodes on one timeline
confirmed_ts = full_df[full_df['class']==1].groupby('time_step').size()
predicted_ts = unknown_df[unknown_df['predicted_label']=='Illicit'].groupby('time_step').size()

all_ts      = sorted(set(confirmed_ts.index) | set(predicted_ts.index))
conf_counts = [confirmed_ts.get(t, 0) for t in all_ts]
pred_counts = [predicted_ts.get(t, 0) for t in all_ts]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(all_ts, conf_counts, color='#FF4444', alpha=0.9,
       label='Confirmed Illicit', width=0.7)
ax.bar(all_ts, pred_counts, color='#FF9900', alpha=0.7,
       label='GNN-Predicted Illicit (from Unknown)',
       bottom=conf_counts, width=0.7)
ax.set_title('Confirmed + GNN-Predicted Illicit Transactions by Time Step\n'
             'Orange = previously unknown wallets now identified as at-risk',
             fontsize=13)
ax.set_xlabel('Time Step')
ax.set_ylabel('Node Count')
ax.legend()

total_conf = sum(conf_counts)
total_pred = sum(pred_counts)
ax.text(0.98, 0.95,
        f'Confirmed illicit : {total_conf:,}\n'
        f'GNN predicted     : {total_pred:,}\n'
        f'Total at-risk     : {total_conf+total_pred:,}\n'
        f'Increase          : {total_pred/total_conf*100:.0f}%',
        transform=ax.transAxes, ha='right', va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='#21262D', alpha=0.9))

plt.tight_layout()
plt.savefig('fig_combined_timeline.png', bbox_inches='tight', dpi=130)
plt.show()

print(f'\nProject summary:')
print(f'  Confirmed illicit  : {total_conf:,}')
print(f'  GNN predicted      : {total_pred:,}')
print(f'  Total at-risk      : {total_conf+total_pred:,}')
print(f'  Detection increase : {total_pred/total_conf*100:.0f}%')
print(f'\n✅ Notebook 04 — Temporal Analysis complete')
print('Next: Run 05_Dashboard_Export.ipynb')